In [4]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import data
from check_submission import check_all

ROOT = Path.cwd()
TRACKS = ["off_market", "list_aware"]
# Walk-forward validation: each of these months is predicted by models trained only
# on the sales before that month, the same way the real test works.
VALIDATION_MONTHS = pd.period_range("2025-12", "2026-05", freq="M")


In [5]:
def validate(
    modelspecs,
    make_model_func=None,
    mode="expanding",
    train_cutoff=None,
    horizon_months=4,
):
    from models.model import make_model

    if make_model_func is None:
        make_model_func = make_model

    listings = data.load("listings")
    results = []
    model_list = []

    if mode == "expanding":
        # Original validation: retrain for every evaluation month
        months = VALIDATION_MONTHS

    elif mode == "fixed_origin":
        # Train once and freeze the model for the entire horizon
        cutoff = pd.Timestamp(train_cutoff)

        if cutoff != cutoff.to_period("M").start_time:
            raise ValueError("train_cutoff must be the first day of a month.")

        if horizon_months < 1:
            raise ValueError("horizon_months must be positive.")

        months = pd.period_range(
            start=cutoff.to_period("M"),
            periods=horizon_months,
            freq="M",
        )

        train = listings.loc[
            listings["date_sold"] < cutoff
        ]

        if train.empty:
            raise ValueError("No training sales before train_cutoff.")

        # Fit each track exactly ONCE
        fixed_models = [
            make_model_func(track, modelspecs=modelspecs).fit(train)
            for track in TRACKS
        ]

        model_list.append(fixed_models)

        print(
            f"Fixed origin: trained on {len(train):,} sales "
            f"before {cutoff.date()}"
        )

    else:
        raise ValueError("mode must be 'expanding' or 'fixed_origin'.")

    for month in months:
        homes = listings.loc[
            listings["date_sold"].dt.to_period("M") == month
        ]

        if mode == "expanding":
            train = listings.loc[
                listings["date_sold"] < month.start_time
            ]

            models = [
                make_model_func(track, modelspecs=modelspecs).fit(train)
                for track in TRACKS
            ]

            model_list.append(models)
        else:
            # Reuse the SAME fitted models for every month
            models = fixed_models

        for track, model in zip(TRACKS, models):
            predicted = model.predict(
                data.hide_answers(homes, track)
            )

            results.append(pd.DataFrame({
                "ml_num": homes["ml_num"],
                "track": track,
                "month": str(month),
                "house_type_name": homes["house_type_name"],
                "sold_price": homes["sold_price"],
                "predicted_price": np.round(predicted),
            }))

        if mode == "expanding":
            print(
                f"{month}: trained on {len(train):,} earlier sales, "
                f"predicted {len(homes):,} homes"
            )
        else:
            print(
                f"{month}: reused frozen models, "
                f"predicted {len(homes):,} homes"
            )

    return pd.concat(results, ignore_index=True), model_list

def bootstrap_ci(values, statistic, repeats=1000):
    """95% interval: recompute the statistic on 1,000 resamples of the homes
    (drawn with replacement) and keep the middle 95% of the results."""
    rng = np.random.default_rng(0)
    results = [statistic(rng.choice(values, size=len(values))) for _ in range(repeats)]
    return [round(float(x), 2) for x in np.percentile(results, [2.5, 97.5])]

def metrics(predictions):
    """Scores for one track. Errors are in percent of the sold price."""
    signed = 100 * (predictions["predicted_price"] - predictions["sold_price"]) / predictions["sold_price"]
    absolute = signed.abs().to_numpy()
    return {
        "n": len(predictions),
        "mape": round(float(np.mean(absolute)), 2),          # mean absolute % error
        "mape_95ci": bootstrap_ci(absolute, np.mean),
        "mdape": round(float(np.median(absolute)), 2),       # median absolute % error
        "mdape_95ci": bootstrap_ci(absolute, np.median),
        "median_signed_error": round(float(np.median(signed)), 2),  # above 0: predicts too high
    }
def report(df):
    # Accept results from earlier notebook runs without refitting the models.
    predictions = pd.concat(df, ignore_index=True) if isinstance(df, list) else df
    scores = {track: metrics(predictions[predictions["track"] == track]) for track in TRACKS}

    (ROOT / "reports").mkdir(exist_ok=True)
    (ROOT / "reports" / "validation_report.json").write_text(json.dumps(scores, indent=2))
    lines = [
        "# Validation report", "",
        f"Walk-forward validation, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}: each month was "
        "predicted by models trained only on the sales before it.",
        "Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.", "",
        "| Track | Homes | MAPE | MdAPE | Median signed error |", "|---|---|---|---|---|"]
    for track, s in scores.items():
        lines.append(f"| {track} | {s['n']:,} | {s['mape']:.2f}% ({s['mape_95ci'][0]:.2f} to {s['mape_95ci'][1]:.2f}) "
                     f"| {s['mdape']:.2f}% ({s['mdape_95ci'][0]:.2f} to {s['mdape_95ci'][1]:.2f}) "
                     f"| {s['median_signed_error']:+.2f}% |")
    lines += [
        "", "MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.",
        "A median signed error above 0 means the model tends to predict too high.",
        "The intervals treat homes as independent. Homes sold in the same month share the same",
        "market swings, so the real uncertainty is somewhat wider.", "",
        "![Error by price range](../figures/error_by_price_range.png)", ""]
    (ROOT / "reports" / "validation_report.md").write_text("\n".join(lines))
    write_latex_table(scores)
    error_by_price_range(predictions)
    print("\n".join(lines))
    print("Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex "
          "and figures/error_by_price_range.png")


def write_latex_table(scores):
    """The same scores as a LaTeX table, for report/report.tex to include."""
    rows = []
    for track, s in scores.items():
        name = track.replace("_", "-")
        rows.append(f"{name} & {s['n']:,} & {s['mdape']:.2f} ({s['mdape_95ci'][0]:.2f}--{s['mdape_95ci'][1]:.2f}) "
                    f"& {s['mape']:.2f} ({s['mape_95ci'][0]:.2f}--{s['mape_95ci'][1]:.2f}) "
                    f"& {s['median_signed_error']:+.2f} \\\\")
    table = "\n".join([
        "% Written by `python run.py report`. Do not edit by hand: rerun the command instead.",
        "\\begin{tabular}{lrccc}", "\\hline",
        "Track & Homes & MdAPE \\% (95\\% CI) & MAPE \\% (95\\% CI) & Median signed error \\% \\\\", "\\hline",
        *rows, "\\hline", "\\end{tabular}"])
    (ROOT / "reports" / "results_table.tex").write_text(table + "\n")


def error_by_price_range(predictions):
    """Figure: how big the errors are, from the cheapest tenth of homes to the dearest.

    Want more figures? Copy this function and group the errors by something else:
    house_type_name, month, or community (add it to the predictions file in validate()).
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colours = {"off_market": "#2a78d6", "list_aware": "#eb6834"}
    for track in TRACKS:
        rows = predictions[predictions["track"] == track]
        error = 100 * (rows["predicted_price"] - rows["sold_price"]).abs() / rows["sold_price"]
        decile = pd.qcut(rows["sold_price"], 10, labels=False) + 1   # 1 = cheapest tenth
        by_decile = error.groupby(decile)
        middle = by_decile.median()
        ax.plot(middle.index, middle.values, marker="o", linewidth=2, color=colours[track], label=track)
        ax.fill_between(middle.index, by_decile.quantile(0.25), by_decile.quantile(0.75),
                        color=colours[track], alpha=0.15, linewidth=0)
    prices = rows["sold_price"].groupby(decile).median()
    ax.set_xticks(prices.index, [f"{d}\n${p / 1e6:.2f}M" for d, p in prices.items()])
    ax.set_xlabel("Sold-price decile (1 = cheapest tenth of homes), with its median price")
    ax.set_ylabel("Absolute % error\n(line: median, band: middle 50%)")
    ax.set_title(f"Validation error by price range, {VALIDATION_MONTHS[0]} to {VALIDATION_MONTHS[-1]}")
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", color="#e5e5e5")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    (ROOT / "figures").mkdir(exist_ok=True)
    fig.savefig(ROOT / "figures" / "error_by_price_range.png", dpi=150)
    plt.close(fig)

In [6]:
from models.boost_regressor import CatBoostRegressor, LightGBMRegressor, XGBoostRegressor
from models.arima import ArimaResidualModel
from models.model import make_model
from models.direct_listing import DirectListingModel
from models.model import OffMarketModel

CATBOOST_PARAMS = {
    "iterations": 900,
    "depth": 6,
    "learning_rate": 0.04,
    "l2_leaf_reg": 5,
    "loss_function": "RMSE",
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}

LIGHTGBM_PARAMS = {
    "n_estimators": 900,
    "learning_rate": 0.04,
    "max_depth": 6,
    "num_leaves": 31,
    "min_child_samples": 40,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,
    "max_bin": 255,
    "objective": "regression",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}

XGBOOST_PARAMS = {
    "n_estimators": 900,
    "learning_rate": 0.04,
    "max_depth": 6,
    "min_child_weight": 5,
    "reg_lambda": 5.0,
    "reg_alpha": 0.0,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "tree_method": "hist",
    "grow_policy": "depthwise",
    "device": "cpu",
    "enable_categorical": True,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
}
0
LGB_MODEL_SPECS = [
    (LightGBMRegressor, "Xpt_spatiotemporal__selected_binary", {'n_estimators': 1800, 'learning_rate': 0.052995139011091236, 'max_depth': 6, 'num_leaves': 40, 'min_child_samples': 30, 'reg_lambda': 16.19372143031597, 'reg_alpha': 0.0, 'subsample': 0.8349167453037107, 'subsample_freq': 1, 'colsample_bytree': 0.953443325579757, 'max_bin': 255, 'objective': 'regression', 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}),
]
XGB_MODEL_SPECS = [
    (XGBoostRegressor, 'Xpt_spatiotemporal', {'n_estimators': 700, 'learning_rate': 0.09522271657253033, 'max_depth': 8, 'min_child_weight': 16.684619386293466, 'reg_lambda': 19.50331236061207, 'reg_alpha': 0.0, 'subsample': 0.9804685587557792, 'colsample_bytree': 0.8994749947027713, 'tree_method': 'hist', 'grow_policy': 'depthwise', 'device': 'cpu', 'enable_categorical': True, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1, 'verbosity': 0}),
]
CAT_MODEL_SPECS = [
    (CatBoostRegressor, 'Xpt_spatiotemporal', {'iterations': 1400, 'depth': 8, 'learning_rate': 0.07127609366218435, 'l2_leaf_reg': 4.14053058084107, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'allow_writing_files': False, 'task_type': 'CPU', 'thread_count': -1, 'random_strength': 0.580328206016201}),
]
MODEL_SPECS = LGB_MODEL_SPECS
def custom_make_model(track, modelspecs=None):
    specs = MODEL_SPECS if modelspecs is None else modelspecs

    if track == "list_aware":
        return DirectListingModel(
            target="ratio",
            modelspecs=specs,
        )

    return OffMarketModel(
        specs,
        center_target=True,
    )


predictions, models = validate(
    modelspecs=MODEL_SPECS,
    make_model_func=custom_make_model,
)

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: house_type_name: object, house_style: object, municipality: object, community: object, garage: object

In [18]:
report(predictions)

# Validation report

Walk-forward validation, 2025-12 to 2026-05: each month was predicted by models trained only on the sales before it.
Errors are (predicted - sold) / sold. Brackets: bootstrap 95% interval.

| Track | Homes | MAPE | MdAPE | Median signed error |
|---|---|---|---|---|
| off_market | 10,069 | 10.65% (10.46 to 10.84) | 8.05% (7.88 to 8.29) | +0.13% |
| list_aware | 10,069 | 5.15% (5.07 to 5.23) | 4.10% (4.00 to 4.19) | +0.57% |

MAPE: mean absolute percentage error. MdAPE: median absolute percentage error.
A median signed error above 0 means the model tends to predict too high.
The intervals treat homes as independent. Homes sold in the same month share the same
market swings, so the real uncertainty is somewhat wider.

![Error by price range](../figures/error_by_price_range.png)

Wrote reports/validation_report.md, reports/validation_report.json, reports/results_table.tex and figures/error_by_price_range.png


In [5]:
# Share these results with notebook.ipynb and the existing report command.
output_path = ROOT / "outputs" / "validation_predictions.csv"
output_path.parent.mkdir(exist_ok=True)
predictions.to_csv(output_path, index=False)

In [20]:
from save_model import save_model
save_model(
    models,
    "CatBoost_Optuna_v1",
    metadata={
        "features": "Xpt_spatiotemporal",
        "tracks": list(TRACKS),
        "validation_months": [str(m) for m in VALIDATION_MONTHS],
        "mode": "expanding",
    },
)


WindowsPath('C:/Users/deant/Desktop/Coding/Data Dash Case Comp/saved_models/CatBoost_Optuna_v1')

In [ ]:
import pandas as pd
import numpy as np

listings = data.load("listings")

origin = pd.Period("2026-02", freq="M")
model_idx = list(VALIDATION_MONTHS).index(origin)

# Reuse the original fitted 900-iteration models
frozen_models = models[model_idx]

results = []

for horizon in range(4):
    month = origin + horizon

    homes = listings.loc[
        listings["date_sold"].dt.to_period("M") == month
    ]

    for track, model in zip(TRACKS, frozen_models):
        predicted = model.predict(
            data.hide_answers(homes, track)
        )

        results.append(pd.DataFrame({
            "ml_num": homes["ml_num"],
            "track": track,
            "month": str(month),
            "horizon": horizon + 1,
            "sold_price": homes["sold_price"],
            "predicted_price": np.round(predicted),
        }))

fixed_predictions = pd.concat(results, ignore_index=True)

fixed_predictions["APE"] = (
    100 * (
        fixed_predictions["predicted_price"]
        - fixed_predictions["sold_price"]
    ).abs()
    / fixed_predictions["sold_price"]
)

print(
    fixed_predictions.groupby(["track", "horizon", "month"])
    ["APE"]
    .agg(
        Homes="size",
        MAPE="mean",
        MdAPE="median",
    )
    .round(2)
)

                            Homes   MAPE  MdAPE
track      horizon month                       
list_aware 1       2026-02   1442   5.44   4.31
           2       2026-03   1816   5.39   4.20
           3       2026-04   2224   5.35   4.31
           4       2026-05   2291   5.57   4.50
off_market 1       2026-02   1442  11.46   8.90
           2       2026-03   1816  11.53   8.84
           3       2026-04   2224  11.81   9.29
           4       2026-05   2291  12.27   9.69


In [ ]:
from models.features import make_features

for name in ["Xpt", "Xpt_spatiotemporal"]:
    features = make_features(
        name,
        model="catboost",
        track="list_aware",
    )

    print(name, "->", type(features.provider).__name__)


list_idx = TRACKS.index("list_aware")
model = models[0][list_idx]

print("Stages:", len(model.components))

component, features = model.components[0]

print("Feature provider:", type(features.provider).__name__)
print("Trained feature names:")
print(component.estimator.feature_names_)

Xpt -> BoostingFeatures
Xpt_spatiotemporal -> RatioComparableFeatures
Stages: 1
Feature provider: RatioComparableFeatures
Trained feature names:
['square_feet', 'bedrooms', 'washrooms', 'parking', 'garage_count', 'lot_front', 'lot_depth', 'lot_area', 'longitude', 'latitude', 'months_since_2021', 'season_sin', 'season_cos', 'log_list_price', 'house_type_name', 'house_style', 'municipality', 'community', 'garage', 'tfidf_0', 'tfidf_1', 'tfidf_2', 'tfidf_3', 'tfidf_4', 'tfidf_5', 'tfidf_6', 'tfidf_7', 'tfidf_8', 'tfidf_9', 'tfidf_10', 'tfidf_11', 'tfidf_12', 'tfidf_13', 'tfidf_14', 'tfidf_15', 'tfidf_16', 'tfidf_17', 'tfidf_18', 'tfidf_19', 'tfidf_20', 'tfidf_21', 'tfidf_22', 'tfidf_23', 'tfidf_24', 'tfidf_25', 'tfidf_26', 'tfidf_27', 'tfidf_28', 'tfidf_29', 'tfidf_30', 'tfidf_31', 'tfidf_32', 'tfidf_33', 'tfidf_34', 'tfidf_35', 'tfidf_36', 'tfidf_37', 'tfidf_38', 'tfidf_39', 'tfidf_40', 'tfidf_41', 'tfidf_42', 'tfidf_43', 'tfidf_44', 'tfidf_45', 'tfidf_46', 'comp_log_price', 'comp_log_pp